# Ordered Logistic Regression Results (FAIRˆ²) Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIRˆ² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")

## 2. Data Overview
Review available record sets with their `@id`s and fields. All entities are shown by `@id` for consistency.

In [ ]:
# List record sets by @id
print("Available record sets (by @id):\n")
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"  - {getattr(rs, '@id', str(rs))} : {getattr(rs, 'name', None)}")
else:
    # Fallback for Croissant 1.0 (record_sets may be empty or not set):
    # Try getting record sets from the internal Dataset object
    # Note: mlcroissant 1.0.x exposes record sets through dataset._record_sets (private)
    # We'll use the API method to get record set ids
    record_set_ids = dataset.record_set_ids
    if record_set_ids:
        for rset_id in record_set_ids:
            print(f"  - {rset_id}")
    else:
        print("No record sets detected in metadata.")

In [ ]:
# Inspect field @ids for a sample record set
# NOTE: Replace '<record_set_id>' with an available record set @id from above, for demonstration we'll use the first available.
record_set_ids = dataset.record_set_ids
if not record_set_ids:
    raise ValueError("No record sets available in dataset.")

# Pick the first record set for demonstration
record_set_id = record_set_ids[0]
print(f"\nFields in record set @id '{record_set_id}':")
# List fields and their @id
record_set_schema = dataset._get_schema(record_set_id)
if record_set_schema and 'field' in record_set_schema:
    for field in record_set_schema['field']:
        print('  -', field.get('@id'), '| Name:', field.get('name'))
else:
    print("No fields found in selected record set.")

## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. All references use `@id`. You can change the `record_set_ids` list to focus on other record sets as needed.

In [ ]:
# Extract data from each record set into a DataFrame, using @id as key
dataframes = {}
for rsid in record_set_ids:
    try:
        # Each record is a dict keyed by field @id
        records = list(dataset.records(record_set=rsid))
        # Create DataFrame using field @id as columns
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records for record set {rsid}")
    except Exception as e:
        print(f"Failed to load record set {rsid}: {str(e)}")

print("\nColumns (@ids) in first record set:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply some key data processing steps using `@id` references. We will select a numeric field and a grouping field for demonstration. Update these values as needed to focus on fields of interest.

In [ ]:
# Example: Pick a numeric field and group-by field from columns (@ids) shown above
# Replace these with the @id of an actual numeric and group field in your dataset

# Let's attempt to auto-detect two such fields as a demo
df = dataframes[record_set_id]

# Find numeric columns by dtype or possible values
numeric_field_id = None
for col in df.columns:
    # Try converting to numeric
    try:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
        converted = pd.to_numeric(df[col])
        if not converted.isnull().all():
            numeric_field_id = col
            break
    except Exception:
        continue

if numeric_field_id is None:
    raise RuntimeError("Could not detect a numeric field @id. Please set 'numeric_field_id' manually.")

print(f"Chosen numeric field @id: {numeric_field_id}")

group_field_id = None
for col in df.columns:
    if col != numeric_field_id and df[col].nunique() < 20 and df[col].nunique() > 1:
        group_field_id = col
        break

if group_field_id:
    print(f"Chosen group-by field @id: {group_field_id}")
else:
    print("No suitable group field auto-detected. Using None.")

# Convert the chosen numeric field to numeric dtype, coercing errors
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter: remove rows with nan
filtered_df = df[df[numeric_field_id].notnull()]

# Use a numeric threshold (mean + std for demo)
threshold = filtered_df[numeric_field_id].mean() + filtered_df[numeric_field_id].std()
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Grouping
if group_field_id and group_field_id in filtered_df.columns:
    group_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_value')
    print(f"\nGrouped data by {group_field_id}:")
    print(group_means.head())

## 5. Visualization
Visualize field distributions and group differences using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, filter, and visualize a Croissant dataset using `mlcroissant`.

- **All dataset schema elements were referenced by their `@id`s** for reproducibility and clarity.
- You can extend these methods by selecting or engineering more features, trying additional record sets (by `@id`), or customizing the data processing for your research or analysis use case.

_If you use this dataset, please cite:_

> Kamadi, V, Chimoita, EL, Wahome, RG and Odhong, C 2026 Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Frontiers.